# 🧪 模块: P7 笔试模拟 02 — Pandas 高危陷阱与避坑专项

本练习专为查漏补缺设计，题目全部源自你的《03_pandas速查手册》里的 `Warning` 和 `Tip` ⚠️。
这是一份“故意挖坑”的卷子，看看能不能闭眼避开这些底层机制陷阱！

## 模块 0: 函数加油站 (Function Cheat Sheet)
这些是你知识库里明确标记过容易踩坑的方法：
- **`select_dtypes` vs `numeric_only`**: 前者选列，后者聚合计。
- **`.item()`**: 从单元素 Series 中安全提取标量，防止对齐出现 NaN。
- **`.clip()` vs 布尔赋值**: 安全截断极值。
- **`include_lowest=True`**: `pd.cut` 时左开闭区间的保命符。
- **`pd.DateOffset` vs `pd.Timedelta`**: 跨月用 Offset，跨天用 Timedelta。


## 模块 1: 概念映射
Pandas 表面上是表格计算，底层全是向量与索引对齐（Index Alignment）。大部分 NaN 报错都源于左右表/Series 的索引不一致。

## 模块 2: 数据准备
*(本节针对语法陷阱验证，使用针对性构造的脏数据集)*

## 模块 3: 查漏补缺挑战 (10 个必踩陷阱)

### Q1: 选列陷阱 (select_dtypes vs numeric_only)
你想提取表里的所有数值列，写 `df_num = df(numeric_only=True)` 报错了。请用最标准的方法提取只包含数字的子数据框。

In [1]:
import pandas as pd
import numpy as np
df = pd.DataFrame({'age': [25, 30], 'name': ['Alice', 'Bob'], 'salary': [10000.0, 20000.0]})
# 你的代码
df_num = df.select_dtypes(include='number').columns
df[df_num]


,age,salary
0,25,10000.0
1,30,20000.0


In [ ]:
# ✅ 参考答案
# 💡 numeric_only 只能用在 .mean() 这类数学聚合算子内部作为参数，不能用来选列！
df_num = df.select_dtypes(include='number')
display(df_num)

### Q2: 字符串正则包含陷阱 (.str.contains)
找出所有包含 'apple' 的行。注意这里混入了一个 NaN，如果不处理会报错 `ValueError`。

In [4]:
import pandas as pd
import numpy as np
df = pd.DataFrame({'desc': ['green apple', 'banana', np.nan, 'red Apple']})
# 你的代码
df[df['desc'].str.contains('apple',case=False,na=False)]

,desc
0,green apple
3,red Apple


In [ ]:
# ✅ 参考答案
# 💡 字符串操作碰到空值极易报错，必须加 na=False，忽略大小写加 case=False
df_apple = df[df['desc'].str.contains('apple', case=False, na=False)]
display(df_apple)

### Q3: 布尔赋值陷阱 (The Boolean Trap)
你想把所有 `age > 90` 的人都强制变成 90 岁。
新手写了: `df['age'] = df.loc[df['age'] > 90] == 90`。结果整列毁了。
请写出正确的原地修改代码（或者用更优雅的 clip 函数）。

In [5]:
import pandas as pd
import numpy as np
df = pd.DataFrame({'name': ['A', 'B'], 'age': [20, 100]})
# 你的代码
df['age'] =np.where(df['age']>90,90,df['age'])
df['age']

0    20
1    90
Name: age, dtype: int64

In [ ]:
# ✅ 参考答案
# 💡 直接赋值会导致整列变成 True/False
# 写法1: df.loc[df['age'] > 90, 'age'] = 90
# 写法2 (更优雅):
df['age'] = df['age'].clip(upper=90)
display(df)

### Q4: 索引对齐导致 NaN 的惨案 (Scalar Extraction)
你分别筛选出了男生的平均分和女生的平均分，得到了两个单行 Series。
你想算他们之间的纯数字差值，直接 `post_val - pre_val` 得到了 NaN（因为 index 不一样）。如何提取纯数字？

In [6]:
import pandas as pd
import numpy as np
post_val = pd.Series([85], index=['M'])  # 模拟筛选结果
pre_val  = pd.Series([80], index=['F'])   # 模拟筛选结果

# 你的代码
diff = post_val.values[0] - pre_val.values[0]
print(diff)

5


In [ ]:
# ✅ 参考答案
# 💡 如果用 Pandas Series 直接相减，会按照 Index = 'M' 和 'F' 对齐，对不上就是 NaN。
# 必须用 .item() 拆开包裹！
diff = post_val.item() - pre_val.item()
print(diff)

### Q5: 时序特征跨组污染防范 (groupby + shift)
计算每个用户相对于自己上一单的购买间隔。
新手写了: `df['date_diff'] = df['date'] - df['date'].shift(1)`。
但这会把用户 B 的第一单减去用户 A 的最后一单！请修复它。

In [12]:
import pandas as pd
import numpy as np
df = pd.DataFrame({
    'uid': ['A', 'A', 'B', 'B'],
    'date': pd.to_datetime(['2023-01-01', '2023-01-05', '2023-01-02', '2023-01-10'])
})
# 你的代码
df.sort_values(by=['uid','date'],inplace=True)
df['pre_date'] = df.groupby('uid')['date'].transform(lambda x: x.shift(1))
df['diff'] = (df['date'] - df['pre_date']).dt.days
df

,uid,date,pre_date,diff
0,A,2023-01-01,NaT,NaN
1,A,2023-01-05,2023-01-01,4.0
2,B,2023-01-02,NaT,NaN
3,B,2023-01-10,2023-01-02,8.0


In [ ]:
# ✅ 参考答案
# 💡 遇到多分组时序特征，必须用 groupby + transform 避免跨组污染
df['date_diff'] = df['date'] - df.groupby('uid')['date'].transform(lambda x: x.shift(1))
display(df)

### Q6: 日期格式化地狱 (Datetime format)
解析英式日期 `'31/12/2023'` (日/月/4位年)。如果只用 `pd.to_datetime()` 可能会搞错月和日。
请用明确的 `format=` 参数来解析。

In [13]:
import pandas as pd
import numpy as np
df = pd.DataFrame({'date_str': ['31/12/2023', '15/01/2023']})
# 你的代码
pd.to_datetime(df['date_str'],format='%d/%m/%Y')

0   2023-12-31
1   2023-01-15
Name: date_str, dtype: datetime64[us]

In [ ]:
# ✅ 参考答案
# 💡 %Y 是4位年，%y是2位年；%m是月，%d是天。
df['date'] = pd.to_datetime(df['date_str'], format='%d/%m/%Y')
display(df)

### Q7: GroupBy Filter 的安检机制
找出那些 **“平均年龄大于 30 岁”的部门** 的 **所有员工记录**。
注意：不是保留 30岁以上的员工，而是“如果部门均龄>30，整个部门保留”。

In [18]:
import pandas as pd
import numpy as np
df = pd.DataFrame({
    'dept': ['Tech', 'Tech', 'HR', 'HR'],
    'name': ['A', 'B', 'C', 'D'],
    'age': [35, 29, 22, 25] 
})
# Info: Tech 组均值 32, HR 组均值 23.5
# 你的代码

df.groupby('dept').filter(lambda x: x['age'].mean() >30)

,dept,name,age
0,Tech,A,35
1,Tech,B,29


In [ ]:
# ✅ 参考答案
# 💡 filter 方法是对“整个组”进行评估并决定是否放行全组行
res = df.groupby('dept').filter(lambda x: x['age'].mean() > 30)
display(res)

### Q8: pd.cut 分箱的左侧致命缺失值
把分数分成 0-60, 60-100。
新手: `pd.cut(df['score'], bins=[0, 60, 100])`。
结果得了 0 分的同学（刚好在 0 边界上）被变成了 NaN！如何修复？

In [21]:
import pandas as pd
import numpy as np
df = pd.DataFrame({'score': [0, 50, 60, 100]})
# 你的代码
pd.cut(df['score'],bins=[0,60,100],include_lowest=True)

0    (-0.001, 60.0]
1    (-0.001, 60.0]
2    (-0.001, 60.0]
3     (60.0, 100.0]
Name: score, dtype: category
Categories (2, interval[float64, right]): [(-0.001, 60.0] < (60.0, 100.0]]

In [ ]:
# ✅ 参考答案
# 💡 pd.cut 默认左开右闭 (0, 60]，所以不包含 0。加上 include_lowest=True 保命！
df['grade'] = pd.cut(df['score'], bins=[0, 60, 100], include_lowest=True, labels=['Fail', 'Pass'])
display(df)

### Q9: 月份和天的加法 (DateOffset vs Timedelta)
给当前日期加上 1 个月，再减去 7 天。
如果是 1月31日加一个月，必须智能跳到 2月底，不能报错。

In [22]:
import pandas as pd
import numpy as np
df = pd.DataFrame({'date': pd.to_datetime(['2023-01-31'])})
# 你的代码
df['date'] + pd.DateOffset(months=1) - pd.DateOffset(days=7)

0   2023-02-21
Name: date, dtype: datetime64[us]

In [ ]:
# ✅ 参考答案
# 💡 月/年用 DateOffset(智能跨月)，天用 Timedelta。
df['next_date'] = df['date'] + pd.DateOffset(months=1) - pd.Timedelta(days=7)
display(df)

### Q10: Query 的变量魔法 (String Magic)
用 `.query()` 找出年龄大于用户输入变量 `min_age` 且 城市在 `target_cities` 列表里的行。

In [23]:
import pandas as pd
import numpy as np
df = pd.DataFrame({'age': [25, 30, 35], 'city': ['BJ', 'SH', 'GZ']})
min_age = 28
target_cities = ['BJ', 'SH']
# 请只使用 .query() 一行代码
df.query('age >= @min_age & city in @target_cities')

,age,city
1,30,SH


In [ ]:
# ✅ 参考答案
# 💡 使用 @ 符号在 query 字符串中引用外部环境变量，代码极度清爽
res = df.query("age > @min_age and city in @target_cities")
display(res)

## 模块 4: 参考答案与复盘
*以上的每道题都在你的 `03_pandas.md` 知识库有原型，如果错了请随时查阅！*